In [1]:
import requests
import pandas as pd
import json

# World Bank API base URL
BASE_URL = "http://api.worldbank.org/v2"

# Indicators we want to extract (директно relevant за credit risk)
INDICATORS = {
    "NY.GDP.PCAP.CD": "gdp_per_capita",      # GDP per capita (current US$)
    "FP.CPI.TOTL.ZG": "inflation_rate",      # Inflation, consumer prices (annual %)
    "SL.UEM.TOTL.ZS": "unemployment_rate"    # Unemployment, total (% of labor force)
}

# Countries — mix от EU, emerging markets, developed
COUNTRIES = [
    "BGR",  # Bulgaria
    "ROU",  # Romania
    "DEU",  # Germany
    "GBR",  # United Kingdom
    "USA",  # United States
    "POL",  # Poland
    "HUN",  # Hungary
    "CZE",  # Czech Republic
    "GRC",  # Greece
    "ITA",  # Italy
    "ESP",  # Spain
    "FRA",  # France
    "TUR",  # Turkey
    "IRL",  # Ireland (Experian HQ!)
    "NLD"   # Netherlands
]

# Date range — последните 15 години
DATE_RANGE = "2010:2024"

print(f"Ready to extract {len(INDICATORS)} indicators for {len(COUNTRIES)} countries")
print(f"Date range: {DATE_RANGE}")

Ready to extract 3 indicators for 15 countries
Date range: 2010:2024


In [7]:
def fetch_worldbank_data(country_code, indicator_code):
    """Fetch data from World Bank API for a specific country and indicator."""
    url = f"{BASE_URL}/country/{country_code}/indicator/{indicator_code}"
    
    params = {
        "format": "json",
        "date": DATE_RANGE,
        "per_page": 100
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"Error for {country_code} - {indicator_code}: Status {response.status_code}")
        return []
    
    data = response.json()
    
    if len(data) < 2 or data[1] is None:
        print(f"No data for {country_code} - {indicator_code}")
        return []
    
    return data[1]


# Test the function
test_data = fetch_worldbank_data("BGR", "NY.GDP.PCAP.CD")

print(f"Got {len(test_data)} records for Bulgaria GDP per capita")
print("\nFirst record sample:")
print(json.dumps(test_data[0], indent=2))

Got 15 records for Bulgaria GDP per capita

First record sample:
{
  "indicator": {
    "id": "NY.GDP.PCAP.CD",
    "value": "GDP per capita (current US$)"
  },
  "country": {
    "id": "BG",
    "value": "Bulgaria"
  },
  "countryiso3code": "BGR",
  "date": "2024",
  "value": 17596.0173663521,
  "unit": "",
  "obs_status": "",
  "decimal": 1
}


In [8]:
import time

# Container for all extracted records
all_records = []

print("Starting extraction...\n")

# Loop through all countries and indicators
for country in COUNTRIES:
    for indicator_code, indicator_name in INDICATORS.items():
        
        # Fetch data
        records = fetch_worldbank_data(country, indicator_code)
        
        # Add metadata to each record
        for record in records:
            all_records.append({
                "country_code": country,
                "country_name": record["country"]["value"],
                "indicator_code": indicator_code,
                "indicator_name": indicator_name,
                "year": int(record["date"]),
                "value": record["value"]
            })
        
        # Small delay to be polite to the API
        time.sleep(0.1)
    
    print(f"✓ {country} completed")

print(f"\n✅ Extraction complete!")
print(f"Total records collected: {len(all_records)}")
print(f"\nSample record:")
print(json.dumps(all_records[0], indent=2))

Starting extraction...

✓ BGR completed
✓ ROU completed
✓ DEU completed
✓ GBR completed
✓ USA completed
✓ POL completed
✓ HUN completed
✓ CZE completed
✓ GRC completed
✓ ITA completed
✓ ESP completed
✓ FRA completed
✓ TUR completed
✓ IRL completed
✓ NLD completed

✅ Extraction complete!
Total records collected: 675

Sample record:
{
  "country_code": "BGR",
  "country_name": "Bulgaria",
  "indicator_code": "NY.GDP.PCAP.CD",
  "indicator_name": "gdp_per_capita",
  "year": 2024,
  "value": 17596.0173663521
}


In [9]:
# Convert list of dictionaries to pandas DataFrame
df = pd.DataFrame(all_records)

# Quick overview
print("=" * 50)
print("DATAFRAME OVERVIEW")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
df.head()

DATAFRAME OVERVIEW

Shape: (675, 6)

Column types:
country_code       object
country_name       object
indicator_code     object
indicator_name     object
year                int64
value             float64
dtype: object

First 5 rows:


,country_code,country_name,indicator_code,indicator_name,year,value
0,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2024,17596.017366
1,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2023,15853.208637
2,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2022,13999.194953
3,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2021,12966.145754
4,BGR,Bulgaria,NY.GDP.PCAP.CD,gdp_per_capita,2020,10760.211975


In [10]:
# Data Quality Checks
print("=" * 50)
print("DATA QUALITY CHECKS")
print("=" * 50)

# 1. Check for missing values
print("\n1. Missing values per column:")
print(df.isnull().sum())

# 2. Check for duplicates
duplicates = df.duplicated().sum()
print(f"\n2. Duplicate rows: {duplicates}")

# 3. Year range validation
print(f"\n3. Year range: {df['year'].min()} - {df['year'].max()}")

# 4. Value statistics per indicator
print("\n4. Value statistics per indicator:")
print(df.groupby('indicator_name')['value'].describe()[['count', 'mean', 'min', 'max']])

# 5. Records per country
print("\n5. Records per country:")
print(df['country_code'].value_counts())

DATA QUALITY CHECKS

1. Missing values per column:
country_code      0
country_name      0
indicator_code    0
indicator_name    0
year              0
value             0
dtype: int64

2. Duplicate rows: 0

3. Year range: 2010 - 2024

4. Value statistics per indicator:
                   count          mean          min            max
indicator_name                                                    
gdp_per_capita     225.0  33471.821222  6853.948055  112894.953241
inflation_rate     225.0      3.925778    -1.735888      72.308836
unemployment_rate  225.0      8.368564     2.015000      27.686000

5. Records per country:
country_code
BGR    45
ROU    45
DEU    45
GBR    45
USA    45
POL    45
HUN    45
CZE    45
GRC    45
ITA    45
ESP    45
FRA    45
TUR    45
IRL    45
NLD    45
Name: count, dtype: int64


In [11]:
# Pivot the data: from long format to wide format
# Long: each row = one country/year/indicator
# Wide: each row = one country/year, columns = indicators

df_wide = df.pivot_table(
    index=['country_code', 'country_name', 'year'],
    columns='indicator_name',
    values='value'
).reset_index()

# Remove the column name "indicator_name" that pandas adds
df_wide.columns.name = None

# Sort for readability
df_wide = df_wide.sort_values(['country_code', 'year'], ascending=[True, False])

print("=" * 50)
print("TRANSFORMED DATAFRAME (WIDE FORMAT)")
print("=" * 50)
print(f"\nShape: {df_wide.shape}")
print(f"\nColumns: {df_wide.columns.tolist()}")
print(f"\nFirst 10 rows:")
df_wide.head(10)

TRANSFORMED DATAFRAME (WIDE FORMAT)

Shape: (225, 6)

Columns: ['country_code', 'country_name', 'year', 'gdp_per_capita', 'inflation_rate', 'unemployment_rate']

First 10 rows:


,country_code,country_name,year,gdp_per_capita,inflation_rate,unemployment_rate
14,BGR,Bulgaria,2024,17596.017366,2.446519,4.200
13,BGR,Bulgaria,2023,15853.208637,9.442841,4.319
12,BGR,Bulgaria,2022,13999.194953,15.325259,4.148
11,BGR,Bulgaria,2021,12966.145754,3.297744,5.166
10,BGR,Bulgaria,2020,10760.211975,1.672441,5.037
9,BGR,Bulgaria,2019,10353.720458,3.103729,4.148
8,BGR,Bulgaria,2018,9849.383972,2.814545,5.211
7,BGR,Bulgaria,2017,8696.689304,2.061596,6.164
6,BGR,Bulgaria,2016,7822.499233,-0.798650,7.575
5,BGR,Bulgaria,2015,7268.654455,-0.104633,9.143


In [12]:
import os

# Create data folder if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Save both formats
# Long format - source of truth for database
df.to_csv('../data/worldbank_long.csv', index=False)

# Wide format - for analytics/BI consumption
df_wide.to_csv('../data/worldbank_wide.csv', index=False)

print("✅ Files saved:")
print(f"   - ../data/worldbank_long.csv  ({len(df)} rows)")
print(f"   - ../data/worldbank_wide.csv  ({len(df_wide)} rows)")

# Verify files
print("\nFiles in data folder:")
for f in os.listdir('../data'):
    size_kb = os.path.getsize(f'../data/{f}') / 1024
    print(f"   {f} ({size_kb:.1f} KB)")

✅ Files saved:
   - ../data/worldbank_long.csv  (675 rows)
   - ../data/worldbank_wide.csv  (225 rows)

Files in data folder:
   worldbank_long.csv (41.7 KB)
   worldbank_wide.csv (13.0 KB)


In [13]:
from sqlalchemy import create_engine, text

# Create SQLite database in data folder
db_path = '../data/worldbank.db'
engine = create_engine(f'sqlite:///{db_path}')

print(f"✅ Database engine created: {db_path}")

# === BUILD DIMENSION TABLES ===

# 1. dim_country — unique countries with metadata
dim_country = df[['country_code', 'country_name']].drop_duplicates().reset_index(drop=True)
dim_country['country_id'] = dim_country.index + 1  # Surrogate key starting from 1

# Add region classification (helpful for SQL analysis later)
region_map = {
    'BGR': 'Eastern Europe', 'ROU': 'Eastern Europe', 'POL': 'Eastern Europe',
    'HUN': 'Eastern Europe', 'CZE': 'Eastern Europe',
    'DEU': 'Western Europe', 'FRA': 'Western Europe', 'NLD': 'Western Europe',
    'GBR': 'Western Europe', 'IRL': 'Western Europe',
    'GRC': 'Southern Europe', 'ITA': 'Southern Europe', 'ESP': 'Southern Europe',
    'TUR': 'Eurasia',
    'USA': 'North America'
}
dim_country['region'] = dim_country['country_code'].map(region_map)

# Reorder columns
dim_country = dim_country[['country_id', 'country_code', 'country_name', 'region']]
print(f"\ndim_country shape: {dim_country.shape}")
print(dim_country)

# 2. dim_indicator — unique indicators with metadata
dim_indicator = df[['indicator_code', 'indicator_name']].drop_duplicates().reset_index(drop=True)
dim_indicator['indicator_id'] = dim_indicator.index + 1

# Add description column
descriptions = {
    'gdp_per_capita': 'Gross Domestic Product per capita in current US dollars',
    'inflation_rate': 'Annual percentage change in consumer price index',
    'unemployment_rate': 'Percentage of total labor force unemployed'
}
dim_indicator['description'] = dim_indicator['indicator_name'].map(descriptions)
dim_indicator = dim_indicator[['indicator_id', 'indicator_code', 'indicator_name', 'description']]

print(f"\ndim_indicator shape: {dim_indicator.shape}")
print(dim_indicator)

✅ Database engine created: ../data/worldbank.db

dim_country shape: (15, 4)
    country_id country_code    country_name           region
0            1          BGR        Bulgaria   Eastern Europe
1            2          ROU         Romania   Eastern Europe
2            3          DEU         Germany   Western Europe
3            4          GBR  United Kingdom   Western Europe
4            5          USA   United States    North America
5            6          POL          Poland   Eastern Europe
6            7          HUN         Hungary   Eastern Europe
7            8          CZE         Czechia   Eastern Europe
8            9          GRC          Greece  Southern Europe
9           10          ITA           Italy  Southern Europe
10          11          ESP           Spain  Southern Europe
11          12          FRA          France   Western Europe
12          13          TUR         Turkiye          Eurasia
13          14          IRL         Ireland   Western Europe
14       

In [14]:
# === BUILD FACT TABLE ===

# Start with the long format DataFrame and JOIN with dimensions
fact_indicators = df.copy()

# Replace country_code with country_id (FK to dim_country)
fact_indicators = fact_indicators.merge(
    dim_country[['country_id', 'country_code']], 
    on='country_code', 
    how='left'
)

# Replace indicator_code with indicator_id (FK to dim_indicator)
fact_indicators = fact_indicators.merge(
    dim_indicator[['indicator_id', 'indicator_code']], 
    on='indicator_code', 
    how='left'
)

# Keep only fact columns (drop the natural keys, keep surrogate keys)
fact_indicators = fact_indicators[['country_id', 'indicator_id', 'year', 'value']]

# Add fact_id as primary key
fact_indicators.insert(0, 'fact_id', range(1, len(fact_indicators) + 1))

print(f"fact_indicators shape: {fact_indicators.shape}")
print(f"\nFirst 5 rows:")
print(fact_indicators.head())

# === LOAD INTO SQLITE ===

print("\n" + "=" * 50)
print("LOADING TO SQLITE")
print("=" * 50)

# Load all 3 tables
dim_country.to_sql('dim_country', engine, if_exists='replace', index=False)
print(f"✅ dim_country loaded: {len(dim_country)} rows")

dim_indicator.to_sql('dim_indicator', engine, if_exists='replace', index=False)
print(f"✅ dim_indicator loaded: {len(dim_indicator)} rows")

fact_indicators.to_sql('fact_indicators', engine, if_exists='replace', index=False)
print(f"✅ fact_indicators loaded: {len(fact_indicators)} rows")

# Verify by querying
print("\n" + "=" * 50)
print("VERIFICATION")
print("=" * 50)

with engine.connect() as conn:
    result = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'"))
    tables = [row[0] for row in result]
    print(f"Tables in database: {tables}")
    
    # Quick row count
    for table in tables:
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table}"))
        count = result.scalar()
        print(f"   {table}: {count} rows")

fact_indicators shape: (675, 5)

First 5 rows:
   fact_id  country_id  indicator_id  year         value
0        1           1             1  2024  17596.017366
1        2           1             1  2023  15853.208637
2        3           1             1  2022  13999.194953
3        4           1             1  2021  12966.145754
4        5           1             1  2020  10760.211975

LOADING TO SQLITE
✅ dim_country loaded: 15 rows
✅ dim_indicator loaded: 3 rows
✅ fact_indicators loaded: 675 rows

VERIFICATION
Tables in database: ['dim_country', 'dim_indicator', 'fact_indicators']
   dim_country: 15 rows
   dim_indicator: 3 rows
   fact_indicators: 675 rows
